# 08. 人機協作 (Human-in-the-Loop)

學習如何在 LangGraph 中實現人機協作流程。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 理解 Human-in-the-Loop (HITL) 的概念
- ✅ 使用 `interrupt_before` 暫停執行
- ✅ 實現人工審核流程
- ✅ 處理人工輸入和修正

---

## 📊 人機協作架構

```
┌─────────────────────────────────────────────────────────┐
│                   人機協作流程                           │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌─────────┐     ┌─────────┐     ┌─────────┐          │
│   │ AI 生成 │ ──▶ │ 人工審核 │ ──▶ │ 執行動作 │          │
│   └─────────┘     └────┬────┘     └─────────┘          │
│                        │                                │
│                   ┌────┴────┐                          │
│                   │ 需要修改？│                          │
│                   └────┬────┘                          │
│                  Yes   │   No                          │
│                   │    │    │                          │
│                   ▼    │    ▼                          │
│              ┌────────┐│ ┌────────┐                    │
│              │ 人工修正││ │ 繼續執行│                    │
│              └────────┘│ └────────┘                    │
│                        │                                │
└────────────────────────┼────────────────────────────────┘
```

### 常見應用場景

| 場景 | 說明 |
|------|------|
| 🛡️ 敏感操作確認 | 刪除資料、發送郵件前確認 |
| ✍️ 內容審核 | AI 生成內容需人工審核 |
| 🔧 錯誤修正 | AI 不確定時請求人工協助 |
| 📋 多步驟審批 | 需要多人簽核的流程 |

In [1]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

---

## 8.1 基本的 Interrupt 機制

### interrupt_before 參數

```python
app = graph.compile(
    checkpointer=memory,
    interrupt_before=["node_name"]  # 在這個節點前暫停
)
```

In [2]:
class ReviewState(TypedDict):
    """審核狀態"""
    content: str           # 待審核內容
    status: str            # pending | approved | rejected
    reviewer_comment: str  # 審核意見

def generate_content(state: ReviewState) -> dict:
    """AI 生成內容"""
    print("  🤖 AI 生成內容...")
    generated = f"AI 生成的內容: 關於 '{state['content']}' 的報告"
    return {"content": generated, "status": "pending"}

def execute_action(state: ReviewState) -> dict:
    """執行動作（需要審核通過）"""
    if state["status"] == "approved":
        print("  ✅ 執行動作: 內容已發布")
        return {"status": "completed"}
    else:
        print("  ❌ 動作被取消")
        return {"status": "cancelled"}

print("✅ 節點定義完成")

✅ 節點定義完成


In [3]:
# 建構圖
graph = StateGraph(ReviewState)
graph.add_node("generate", generate_content)
graph.add_node("execute", execute_action)

graph.add_edge(START, "generate")
graph.add_edge("generate", "execute")
graph.add_edge("execute", END)

# 🔑 關鍵：在 execute 前暫停等待人工審核
memory = MemorySaver()
app = graph.compile(
    checkpointer=memory,
    interrupt_before=["execute"]  # ← 在執行前暫停
)

print("✅ 帶人工審核的圖編譯完成")
print("💡 執行會在 'execute' 節點前暫停")

✅ 帶人工審核的圖編譯完成
💡 執行會在 'execute' 節點前暫停


In [4]:
print("📊 圖結構:")
print(app.get_graph().draw_mermaid())

📊 圖結構:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	execute(execute<hr/><small><em>__interrupt = before</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	generate --> execute;
	execute --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



---

## 8.2 模擬人機協作流程

In [5]:
config = {"configurable": {"thread_id": "review-1"}}

print("🔄 人機協作流程示範：")
print("=" * 50)

# 步驟 1: 執行到中斷點
print("\n📍 步驟 1: 執行到中斷點")
result = app.invoke(
    {"content": "產品發布公告", "status": "", "reviewer_comment": ""},
    config
)
print(f"   當前狀態: {result['status']}")
print(f"   生成內容: {result['content'][:40]}...")

🔄 人機協作流程示範：

📍 步驟 1: 執行到中斷點
  🤖 AI 生成內容...
   當前狀態: pending
   生成內容: AI 生成的內容: 關於 '產品發布公告' 的報告...


In [6]:
# 步驟 2: 模擬人工審核（批准）
print("\n📍 步驟 2: 人工審核")
print("   👤 審核員檢視內容...")
print("   👤 審核員決定: 批准 ✅")

# 更新狀態（模擬人工輸入）
app.update_state(
    config,
    {"status": "approved", "reviewer_comment": "內容符合規範"}
)

# 繼續執行
print("\n📍 步驟 3: 繼續執行")
final_result = app.invoke(None, config)  # None 表示繼續
print(f"   最終狀態: {final_result['status']}")


📍 步驟 2: 人工審核
   👤 審核員檢視內容...
   👤 審核員決定: 批准 ✅

📍 步驟 3: 繼續執行
  ✅ 執行動作: 內容已發布
   最終狀態: completed


---

## 8.3 帶拒絕選項的審核流程

In [7]:
config_reject = {"configurable": {"thread_id": "review-2"}}

print("🔄 審核拒絕流程示範：")
print("=" * 50)

# 執行到中斷點
print("\n📍 步驟 1: 執行到中斷點")
result = app.invoke(
    {"content": "爭議性內容", "status": "", "reviewer_comment": ""},
    config_reject
)

# 模擬人工審核（拒絕）
print("\n📍 步驟 2: 人工審核")
print("   👤 審核員決定: 拒絕 ❌")

app.update_state(
    config_reject,
    {"status": "rejected", "reviewer_comment": "內容不符合社群規範"}
)

# 繼續執行
print("\n📍 步驟 3: 繼續執行")
final_result = app.invoke(None, config_reject)
print(f"   最終狀態: {final_result['status']}")
print(f"   審核意見: {final_result['reviewer_comment']}")

🔄 審核拒絕流程示範：

📍 步驟 1: 執行到中斷點
  🤖 AI 生成內容...

📍 步驟 2: 人工審核
   👤 審核員決定: 拒絕 ❌

📍 步驟 3: 繼續執行
  ❌ 動作被取消
   最終狀態: cancelled
   審核意見: 內容不符合社群規範


---

## 8.4 多步驟審核流程

In [8]:
class MultiReviewState(TypedDict):
    """多步驟審核狀態"""
    content: str
    step: int
    approvals: Annotated[list, lambda a, b: a + b]
    final_status: str

def step_1_review(state):
    print("  📋 第一階段審核...")
    return {"step": 1}

def step_2_review(state):
    print("  📋 第二階段審核...")
    return {"step": 2}

def finalize(state):
    approvals = state.get("approvals", [])
    if len(approvals) >= 2 and all(a["approved"] for a in approvals):
        status = "approved"
    else:
        status = "rejected"
    print(f"  ✅ 最終結果: {status}")
    return {"final_status": status}

# 建構多步驟審核圖
multi_graph = StateGraph(MultiReviewState)
multi_graph.add_node("step1", step_1_review)
multi_graph.add_node("step2", step_2_review)
multi_graph.add_node("finalize", finalize)

multi_graph.add_edge(START, "step1")
multi_graph.add_edge("step1", "step2")
multi_graph.add_edge("step2", "finalize")
multi_graph.add_edge("finalize", END)

multi_memory = MemorySaver()
multi_app = multi_graph.compile(
    checkpointer=multi_memory,
    interrupt_before=["step2", "finalize"]  # 多個中斷點
)

print("✅ 多步驟審核流程就緒")

✅ 多步驟審核流程就緒


In [9]:
config = {"configurable": {"thread_id": "multi-review-1"}}

print("🔄 多步驟審核示範：")
print("=" * 50)

# 步驟 1
print("\n--- 第一次執行（到 step1）---")
result = multi_app.invoke(
    {"content": "重要合約", "step": 0, "approvals": [], "final_status": ""},
    config
)

# 第一階段審核
print("\n--- 第一階段審核 ---")
multi_app.update_state(config, {"approvals": [{"reviewer": "A", "approved": True}]})
result = multi_app.invoke(None, config)

# 第二階段審核
print("\n--- 第二階段審核 ---")
multi_app.update_state(config, {"approvals": [{"reviewer": "B", "approved": True}]})
result = multi_app.invoke(None, config)

print("\n" + "=" * 50)
print(f"審核通過數: {len(result.get('approvals', []))}")
print(f"最終結果: {result['final_status']}")

🔄 多步驟審核示範：

--- 第一次執行（到 step1）---
  📋 第一階段審核...

--- 第一階段審核 ---
  📋 第二階段審核...

--- 第二階段審核 ---
  ✅ 最終結果: approved

審核通過數: 2
最終結果: approved


---

## 💡 重點回顧

### HITL API 總結

```python
# 設置中斷點
app = graph.compile(
    checkpointer=memory,
    interrupt_before=["node"],  # 節點前暫停
    interrupt_after=["node"]    # 節點後暫停
)

# 更新狀態（人工輸入）
app.update_state(config, {"key": "value"})

# 繼續執行
result = app.invoke(None, config)
```

### 設計原則

| 原則 | 說明 |
|------|------|
| 明確中斷點 | 選擇關鍵決策點暫停 |
| 狀態可讀 | 確保人工能理解當前狀態 |
| 可恢復 | 使用 checkpointer 保存進度 |
| 超時處理 | 設計等待超時的處理邏輯 |

---

## 📝 練習題

1. **工具確認**：AI 呼叫工具前請求確認
2. **修改重試**：審核拒絕後允許修改並重新提交
3. **超時機制**：等待超過 5 分鐘自動取消
4. **審核隊列**：多個任務排隊等待審核

---

下一步：[09. 多 Agent 系統](09_multi_agent.ipynb)